<a href="https://colab.research.google.com/github/RA-1020/tiny-wfm-backbone/blob/main/notebooka30bfb4126.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
aleksandrdubrovin_deepsigio_radioml_201801a_new_path = kagglehub.dataset_download('aleksandrdubrovin/deepsigio-radioml-201801a-new')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import numpy as np

signals = np.load("/kaggle/input/datasets/aleksandrdubrovin/deepsigio-radioml-201801a-new/signals.npy")
labels  = np.load("/kaggle/input/datasets/aleksandrdubrovin/deepsigio-radioml-201801a-new/labels.npy")
snrs    = np.load("/kaggle/input/datasets/aleksandrdubrovin/deepsigio-radioml-201801a-new/snrs.npy")

print("signals:", signals.shape, signals.dtype)
print("labels:", labels.shape, labels.dtype, "unique:", np.unique(labels)[:10])
print("snrs:", snrs.shape, snrs.dtype, "unique:", np.unique(snrs))

with open("/kaggle/input/datasets/aleksandrdubrovin/deepsigio-radioml-201801a-new/classes.txt") as f:
    print("classes.txt:\n", f.read())

In [ ]:
import numpy as np
import os

BASE = "/kaggle/input/datasets/aleksandrdubrovin/deepsigio-radioml-201801a-new"

signals = np.load(f"{BASE}/signals.npy", mmap_mode="r")   # <- memory-mapped, not loaded into RAM
labels  = np.load(f"{BASE}/labels.npy")                   # small enough (~245MB) to load fully
snrs    = np.load(f"{BASE}/snrs.npy")[:, 0]

with open(f"{BASE}/classes.txt") as f:
    exec(f.read())

Y = np.argmax(labels, axis=1)

KEEP_CLASSES = ['OOK','BPSK','QPSK','8PSK','16QAM','64QAM','AM-DSB-WC','FM','GMSK','OQPSK']
KEEP_SNRS    = [10, 14, 18]
PER_CLASS_SNR = 400

keep_idx = [classes.index(c) for c in KEEP_CLASSES]

Xs, ys = [], []
for new_label, ci in enumerate(keep_idx):
    for snr in KEEP_SNRS:
        idx = np.where((Y == ci) & (snrs == snr))[0][:PER_CLASS_SNR]
        Xs.append(np.array(signals[idx]))   # pulls ONLY these rows off disk into RAM
        ys.append(np.full(len(idx), new_label))

X_sub = np.concatenate(Xs)
y_sub = np.concatenate(ys)
print(X_sub.shape, y_sub.shape)

np.savez_compressed("radioml_subset.npz",
                    X=X_sub, y=y_sub,
                    classes=np.array(KEEP_CLASSES))
print("saved:", os.path.getsize("radioml_subset.npz")/1e6, "MB")